In [ ]:
import pymysql
import requests
import time
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline
import nltk

# Download VADER lexicon
nltk.download('vader_lexicon')

# Initialize VADER and Hugging Face sentiment analysis
sia = SentimentIntensityAnalyzer()
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# MySQL connection setup
def setup_mysql_connection():
    """
    Establishes a connection to the MySQL database.
    """
    connection = pymysql.connect(
        host="LOCALHOST",          # Replace with your MySQL host (e.g., "127.0.0.1")
        user="USERNAME",      # Replace with your MySQL username
        password="PASSWORD",  # Replace with your MySQL password
        database="DB NAME"  # Replace with your MySQL database name
    )
    return connection

# Save sentiment results to MySQL database
def save_results_to_mysql(connection, rows):
    """
    Inserts sentiment analysis results into the database.
    """
    try:
        cursor = connection.cursor()
        query = '''
            INSERT INTO SentimentResults (
                headline, topic, textblob_sentiment, textblob_score,
                vader_sentiment, vader_score,
                huggingface_sentiment, huggingface_score
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        '''
        cursor.executemany(query, rows)
        connection.commit()
        print(f"{cursor.rowcount} rows inserted successfully!")
    except pymysql.MySQLError as e:
        print(f"Error inserting data into MySQL: {e}")

# Analyze sentiment using TextBlob
def analyze_sentiment_textblob(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    sentiment = 'Positive' if polarity > 0 else 'Negative' if polarity < 0 else 'Neutral'
    return polarity, sentiment

# Analyze sentiment using VADER
def analyze_sentiment_vader(text):
    scores = sia.polarity_scores(text)
    compound_score = scores['compound']
    sentiment = 'Positive' if compound_score >= 0.05 else 'Negative' if compound_score <= -0.05 else 'Neutral'
    return compound_score, sentiment

# Analyze sentiment using Hugging Face Transformers
def analyze_sentiment_huggingface(text):
    result = sentiment_pipeline(text)[0]
    return result['score'], result['label']

# Fetch headlines by topic
def fetch_headlines_by_topic(api_key, topic):
    """
    Fetches news headlines for a specific topic using the NewsAPI.
    """
    url = f'https://newsapi.org/v2/top-headlines?country=us&category={topic}&apiKey={api_key}'
    response = requests.get(url)
    if response.status_code == 200:
        articles = response.json().get('articles', [])
        return [article['title'] for article in articles]
    else:
        print(f"Error fetching news for topic '{topic}': {response.status_code}")
        return []

# Analyze headlines for a specific topic
def analyze_topic_sentiment(api_key, topic, connection):
    """
    Fetches headlines for a topic, performs sentiment analysis, and stores the results in MySQL.
    """
    print(f"\nFetching headlines for topic: {topic}")
    headlines = fetch_headlines_by_topic(api_key, topic)
    
    if not headlines:
        print(f"No headlines found for topic: {topic}")
        return

    rows = []

    # Perform sentiment analysis for each headline
    for headline in headlines:
        tb_score, tb_sentiment = analyze_sentiment_textblob(headline)
        vader_score, vader_sentiment = analyze_sentiment_vader(headline)
        hf_score, hf_sentiment = analyze_sentiment_huggingface(headline)

        # Append results to rows
        rows.append((headline, topic, tb_sentiment, tb_score,
                     vader_sentiment, vader_score,
                     hf_sentiment, hf_score))

    # Save results to MySQL database
    save_results_to_mysql(connection, rows)
    print(f"Inserted {len(rows)} rows into the database for topic '{topic}'.")

# Main function
def main():
    """
    Main function to run the sentiment analysis pipeline with topic categorization.
    """
    api_key = "e867900d8528496fb5c2ba8fe1a6390b"  # Replace with your NewsAPI.org API key
    connection = setup_mysql_connection()  # Set up MySQL connection

    # Topics to analyze
    topics = ["technology", "sports", "business", "entertainment", "health"]

    while True:
        for topic in topics:
            analyze_topic_sentiment(api_key, topic, connection)

        print("\nWaiting for 5 minutes before fetching the next batch...")
        time.sleep(300)  # Wait for 5 minutes

if __name__ == "__main__":
    main()
